# Logistic-regression gradients with autograd

Let PyTorch's autograd do the calculus for a logistic-regression loss, and read gradients for both the parameters and an intermediate tensor.

Write `logistic_grads(x, y, w, b)`:

- `x`: float64 tensor `(N, D)`; `y`: float64 tensor `(N,)` of 0/1 labels.
- `w`: float64 tensor `(D,)`; `b`: Python float.
- Compute the logits $z = x w + b$ (shape `(N,)`) and the mean binary cross-entropy

$$L = -\frac{1}{N}\sum_{n} \Big[y_n \log \sigma(z_n) + (1-y_n)\log\big(1-\sigma(z_n)\big)\Big].$$

Return a tuple `(loss, grad_w, grad_b, grad_z)`:

- `loss`: 0-d tensor (or float).
- `grad_w` `(D,)`, `grad_b` 0-d, `grad_z` `(N,)`: the gradients $\partial L/\partial w$, $\partial L/\partial b$, $\partial L/\partial z$.

Get all of them from a single `backward()` call. Don't modify the input `w`. Large $|z|$ must not produce `inf` or `nan`, so compute the loss from the logits, not from $\sigma(z)$.

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import torch

def logistic_grads(x, y, w, b):
    """Return (loss, dL/dw, dL/db, dL/dz)."""
    w = w.clone().requires_grad_()
    z = x @ w + b
    loss = torch.nn.functional.binary_cross_entropy(torch.sigmoid(z), y)
    loss.backward()
    return loss, w.grad, torch.tensor(0.0), torch.zeros_like(z)


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[1.0, 0.0], [0.0, 1.0]], "dtype": "float64"}, {"__nd__": [1.0, 0.0], "dtype": "float64"}, {"__nd__": [0.5, -0.5], "dtype": "float64"}, 0.0], "e": {"__tup__": [{"__nd__": 0.4740769841801067, "dtype": "float64"}, {"__nd__": [-0.1887703343990727, 0.1887703343990727], "dtype": "float64"}, {"__nd__": 0.0, "dtype": "float64"}, {"__nd__": [-0.1887703343990727, 0.1887703343990727], "dtype": "float64"}]}}, {"a": [{"__nd__": [[2.0], [-1.0], [0.5]], "dtype": "float64"}, {"__nd__": [1.0, 1.0, 0.0], "dtype": "float64"}, {"__nd__": [1.0], "dtype": "float64"}, 0.25], "e": {"__tup__": [{"__nd__": 0.7913161903821823, "dtype": "float64"}, {"__nd__": [0.27602303965495684], "dtype": "float64"}, {"__nd__": -0.0317831549663698, "dtype": "float64"}, {"__nd__": [-0.03178315496636982, -0.226392899725131, 0.226392899725131], "dtype": "float64"}]}}, {"a": [{"__nd__": [[0.03419276725318417, 1.3597475403099617, 1.2247210785859324], [-0.5103070767876675, -0.2979695111064471, -0.5273841930334252], [0.5697263575719601, -0.056064439045617594, 0.7468856162565439], [-1.8473247989741095, 1.5665487746995206, -0.09643216015562055], [0.6803784532741461, -0.13656633397682774, -0.3790985670748533], [0.46311015859758675, 0.824513527530113, -0.20252987069345152], [-0.15278617857019708, 0.685698610809258, -0.8703406419471712], [-1.5143835037313955, 0.39498186274953, -0.6705658236878794]], "dtype": "float64"}, {"__nd__": [0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0], "dtype": "float64"}, {"__nd__": [-0.7435960295088448, 0.3849938087479083, 0.7172358071943838], "dtype": "float64"}, -0.3], "e": {"__tup__": [{"__nd__": 0.7989140807361181, "dtype": "float64"}, {"__nd__": [-0.05474222327206992, 0.2279790186144983, 0.131768297370438], "dtype": "float64"}, {"__nd__": 0.1301765382000379, "dtype": "float64"}, {"__nd__": [0.09322847114959956, -0.07524123915680417, 0.055975803146744524, -0.02086508088625791, -0.09448924966235434, 0.04801207803056589, 0.045830580056850985, 0.07772517552169336], "dtype": "float64"}]}}, {"a": [{"__nd__": [[1.0], [1.0]], "dtype": "float64"}, {"__nd__": [0.0, 1.0], "dtype": "float64"}, {"__nd__": [0.0], "dtype": "float64"}, 0.0], "e": {"__tup__": [{"__nd__": 0.6931471805599453, "dtype": "float64"}, {"__nd__": [0.0], "dtype": "float64"}, {"__nd__": 0.0, "dtype": "float64"}, {"__nd__": [0.25, -0.25], "dtype": "float64"}]}}, {"a": [{"__nd__": [[100.0], [-100.0]], "dtype": "float64"}, {"__nd__": [0.0, 1.0], "dtype": "float64"}, {"__nd__": [10.0], "dtype": "float64"}, 0.0], "e": {"__tup__": [{"__nd__": 1000.0, "dtype": "float64"}, {"__nd__": [100.0], "dtype": "float64"}, {"__nd__": 0.0, "dtype": "float64"}, {"__nd__": [0.5, -0.5], "dtype": "float64"}]}}, {"a": [{"__nd__": [[-0.3000105984884774, 0.5446678079208929, 1.0428754765829538, -0.20695643620832396, -0.8135155419815723]], "dtype": "float64"}, {"__nd__": [1.0], "dtype": "float64"}, {"__nd__": [0.3476505985155095, 0.24754574096284754, 1.0988127684144084, -1.284580778805345, -0.6616129303555477], "dtype": "float64"}, 2.0], "e": {"__tup__": [{"__nd__": 0.0185032887404909, "dtype": "float64"}, {"__nd__": [0.005500140462026807, -0.009985478725759772, -0.01911919657008283, 0.0037941641875361646, 0.014914305599481563], "dtype": "float64"}, {"__nd__": -0.018333153861022855, "dtype": "float64"}, {"__nd__": [-0.018333153861022855], "dtype": "float64"}]}}, {"a": [{"__nd__": [[-0.8381669607156745, -1.7340148462328515, 0.1264345551969962, 0.527804212495524], [-0.7387900314758065, 1.3856470744961586, 0.8219243366604353, 0.6273764788355353], [0.4017070914409699, 0.955669564448635, -1.3319798395431022, 0.6139296582498643], [0.6027768335334479, -1.7677185771429749, 0.34703010205437973, -0.2504213467099684], [0.7815226960616993, -0.4390621876376686, -0.01824085764910033, 0.3428515173176555], [-0.8762616887442077, 0.5985966481803844, -0.10496318852366823, 0.49248262367924284], [-0.5218375063367878, 1.0862015432775176, 0.6052019784294742, -0.17802502471933673], [0.6319571570936101, 1.259755161358625, 1.7911755134979888, -1.5735763704402195], [0.8831318116225195, 0.4650685085133813, -0.09386078018634399, -1.0066649349770713], [1.2571886134731436, -1.2617379934445705, 0.5669454657347489, 1.3018679962026896], [-1.5996692880514796, -0.30251784048326236, -1.3092168175162993, 0.24405410803590055], [1.5143751306746547, 2.0235560291721977, -1.7781144428835152, -0.5749490057210601], [0.7035450933309114, 1.5793726520932216, 0.4212106442782976, -0.7461519975907094], [0.2971315096374385, -0.01661920098083325, -0.2037407223881694, -0.7344710985474203], [0.38725975594300555, 0.3078796263658112, -0.09298412757635025, -0.2216879397006701], [-1.284916570539673, -0.48617566714143334, 1.2064496713466424, -0.19055825686602823], [-1.4396973168381253, 1.3344428069513163, 0.5302654084829203, 2.1080870286253033], [0.06251216939171997, -0.46138457905169217, -1.4476446987755465, 1.3238383700638794], [2.5695115129905117, -0.820934937223875, -0.6470876402241984, 0.5961887986723416], [-0.8303454354133867, -0.27055836699218927, -0.34983769944206944, 0.19195888729205715], [1.09481844340068, 0.022067589713919693, 0.9189100957801388, -0.41988889600049356], [0.3277985357234032, -2.138225589513189, -1.4499480667813884, 0.7959134126817742], [-0.590149399040946, 0.5799149234726574, 0.5423442548146441, 1.3222788582368146], [0.8118590596762011, 1.0169913501666112, -0.11167133066420938, -0.6982851765628781], [-0.731558777725664, -0.4880439402887327, -1.1298291140131056, -0.5474435821203582], [-0.09256905150847415, 0.2516119571058145, -0.33889079006978534, -1.923682866353282], [-0.07228295450292373, 0.22534578702721786, 1.084475594643844, 0.5778638956158266], [-0.6435606861455095, -0.7237765601247179, 2.0105975223757233, 0.756637954017235], [1.831439822737841, 2.129444531219653, -0.818083666605393, 0.38526930227745887], [0.4581670048370468, 0.5596122240618859, 0.5419022252820672, 0.20205463691158088]], "dtype": "float64"}, {"__nd__": [0.0, 1.0, 0.0, 1.0, 1.0, 0.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 1.0], "dtype": "float64"}, {"__nd__": [3.9359499758025374, 0.13277498591182302, -3.5025908548338296, 0.025132209532262292], "dtype": "float64"}, 1.0], "e": {"__tup__": [{"__nd__": 1.9553749823494981, "dtype": "float64"}, {"__nd__": [0.2126560337909707, -0.02305808107101683, -0.3149300998058679, -0.13851264848689593], "dtype": "float64"}, {"__nd__": -0.16848830167579634, "dtype": "float64"}, {"__nd__": [0.0016440737454787742, -0.0329972960035761, 0.03331274144176224, -0.004276970526988759, -0.0005486561301611255, 0.004010596508751313, 0.0015305086181600714, -0.031151479195501725, -0.0002611978289872622, -0.0007099117722785709, -0.02260003554329065, -4.839163526559887e-08, 0.030768832695845353, -0.0018013778391997692, -0.0017629555356516594, -0.03332547425508499, -0.03327185104587015, 0.03327153385246649, -5.6606895800411415e-08, -0.02484496804139947, -0.003692130178900821, -2.734732593944189e-05, -0.031913077086345734, -0.0002992377227152806, -0.00397555953559973, 0.028636162637819085, -0.03180949595093844, -0.033327510820743735, -3.859737432221344e-07, -0.00906572786463196], "dtype": "float64"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(logistic_grads, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")